# 03 - Risk-aware decision engine

The engine encodes a metric hierarchy modeled on Microsoft ExP / Spotify practice:

| role | metric | test | multiplicity | CUPED covariate |
|---|---|---|---|---|
| primary success | checkout conversion | one-sided superiority z-test | BH-corrected | `pre_exp_spend` (-19% variance) |
| secondary success | revenue per user | one-sided superiority z-test | BH-corrected | `pre_exp_spend` (-39% variance) |
| guardrail | average order value | non-inferiority vs -2% margin | uncorrected, power reported | none: unsafe behind a post-treatment filter |
| guardrail | checkout error rate | non-inferiority vs +0.5pp margin | uncorrected, power reported | none: no useful covariate exists |

Before any of this the engine runs a **sample ratio mismatch (SRM) check**: a chi-square goodness-of-fit test on the assignment split, gated at p < 0.0005. If it fires, the arms are not comparable and the engine returns **INVALID** and reports no metrics at all -- see section 4.

**Why do the guardrails get no covariate?** CUPED's unbiasedness rests on `E[x | treatment] = E[x | control]`, which randomization guarantees only on the full randomized population. Average order value is measured on converters only, and treatment changes *who* converts, so within that subgroup the arms' covariate means genuinely differ and the adjustment injects bias instead of removing noise. Measured on this generator with a true AOV effect of exactly **zero**, converters-only CUPED returns **+$0.70** against a $1.18 margin (12 Monte Carlo standard errors from zero over 400 simulations) while the raw estimator stays unbiased -- and it biases *toward passing*, the worst direction for a safety check. `MetricSpec` refuses the combination rather than trusting a reader to remember. It is the same failure mode as the original AOV incident: conditioning on a post-treatment variable.

**Why one-sided superiority?** We ship only on improvement; "no effect" and "harm" both mean do-not-ship, so they share the null.

**Why is BH applied to success metrics only?** For success metrics the costly error is a FALSE claim of improvement (Type I), which multiplies with the number of metrics, so the family is FDR-corrected. For guardrails the costly error is MISSING real harm (Type II): alpha-correcting them would make harm even easier to miss. Instead each guardrail reports its power to clear the margin, and low power triggers INVESTIGATE rather than a silent pass.

**Verdicts**: SHIP (primary significant after BH + every guardrail non-inferior with adequate power), DO NOT SHIP (primary fails or harm beyond a margin is established), INVESTIGATE (a guardrail is inconclusive).

Note on the guardrails: a statistically significant degradation SMALLER than the margin does not block shipping. The margin is the business's declared tolerance; with big samples even accepted, trivial costs become statistically significant.

In [1]:
import sys
sys.path.insert(0, "..")  # so config, src and data import from the repo root

import config
from data.generate_data import generate_experiment_data
from src.power import required_sample_size_per_arm
from src.decision_engine import default_metric_specs, evaluate_experiment

n_default = required_sample_size_per_arm(
    config.CONTROL_CONVERSION_RATE, config.MDE_RELATIVE
)

## 1. The experiment as designed in notebook 01: verdict is INVESTIGATE

We first run the engine on the experiment at the sample size that powers the PRIMARY metric (70k per arm). The result is a finding in itself: the sample size that suffices for conversion leaves the average order value guardrail with only ~20% power, because AOV lives on the ~3.5% converter subpopulation and order values are heavy-tailed. Relative to what the guardrails need, the primary-powered sample size IS the underpowered case.

In [2]:
df = generate_experiment_data(n_users=2 * n_default, seed=204)
print(evaluate_experiment(df, default_metric_specs()).render())

# Decision report

## VERDICT: INVESTIGATE

- primary improved, but at least one guardrail is inconclusive

alpha = 0.05, guardrail power floor = 80%

### sample ratio check (ok)
- treatment 70,229 / control 70,229 = 50.0000% vs 50.00% intended
- chi-square = 0.000, p = 1

### checkout conversion rate (success, proportion test)
- effect: +0.00257 (one-sided 95% lower bound +0.00110)
- one-sided superiority p = 0.0020; significant after BH: YES
- CUPED on 'pre_exp_spend': 18.9% of metric variance removed
- status: PASS

### revenue per user (success, mean test)
- effect: +0.11295 (one-sided 95% lower bound +0.01258)
- one-sided superiority p = 0.0321; significant after BH: YES
- CUPED on 'pre_exp_spend': 39.0% of metric variance removed
- status: PASS

### average order value (guardrail, mean test)
- effect: -0.15327, margin: -1.17951 (relative 0.02), worst-case bound: -2.52879
- non-inferiority p = 0.2387 (NOT established); power to clear margin if harmless = 20%
- status: INCONCLUSIVE

## 2. SHIP scenario

To power the AOV guardrail to ~90% against its 2% margin we need roughly 900k users per arm (heavy-tailed order values, converters-only population). We simulate a variant with no true AOV change at that traffic level: primary improves, both guardrails clear their margins with adequate power. Note the error-rate guardrail: the true +0.1pp degradation is statistically detected, but it is WITHIN the +0.5pp margin the business accepted, so it does not block the ship.

In [3]:
df_ship = generate_experiment_data(n_users=1_800_000, seed=202, aov_effect=0.0)
print(evaluate_experiment(df_ship, default_metric_specs()).render())

# Decision report

## VERDICT: SHIP

- primary metric 'checkout conversion rate' significantly improved after BH correction
- all guardrails non-inferior with adequate power

alpha = 0.05, guardrail power floor = 80%

### sample ratio check (ok)
- treatment 900,000 / control 900,000 = 50.0000% vs 50.00% intended
- chi-square = 0.000, p = 1

### checkout conversion rate (success, proportion test)
- effect: +0.00283 (one-sided 95% lower bound +0.00241)
- one-sided superiority p = 0.0000; significant after BH: YES
- CUPED on 'pre_exp_spend': 17.8% of metric variance removed
- status: PASS

### revenue per user (success, mean test)
- effect: +0.17141 (one-sided 95% lower bound +0.14294)
- one-sided superiority p = 0.0000; significant after BH: YES
- CUPED on 'pre_exp_spend': 36.8% of metric variance removed
- status: PASS

### average order value (guardrail, mean test)
- effect: +0.26458, margin: -1.17519 (relative 0.02), worst-case bound: -0.38466
- non-inferiority p = 0.0001 (established

## 3. DO NOT SHIP scenario

Same traffic, but the true average order value effect is cranked to -5%, far past the -2% margin. The engine now establishes harm beyond the margin (the entire CI sits outside it) and blocks the ship even though the primary metric improved: exactly the trade-off guardrails exist to catch (the streamlined checkout converts more users but they buy noticeably less per order).

In [4]:
df_harm = generate_experiment_data(n_users=1_800_000, seed=203, aov_effect=-0.05)
print(evaluate_experiment(df_harm, default_metric_specs()).render())

# Decision report

## VERDICT: DO NOT SHIP

- harm beyond a guardrail margin is statistically established

alpha = 0.05, guardrail power floor = 80%

### sample ratio check (ok)
- treatment 900,000 / control 900,000 = 50.0000% vs 50.00% intended
- chi-square = 0.000, p = 1

### checkout conversion rate (success, proportion test)
- effect: +0.00229 (one-sided 95% lower bound +0.00188)
- one-sided superiority p = 0.0000; significant after BH: YES
- CUPED on 'pre_exp_spend': 18.8% of metric variance removed
- status: PASS

### revenue per user (success, mean test)
- effect: +0.03195 (one-sided 95% lower bound +0.00410)
- one-sided superiority p = 0.0296; significant after BH: YES
- CUPED on 'pre_exp_spend': 37.5% of metric variance removed
- status: PASS

### average order value (guardrail, mean test)
- effect: -2.49843, margin: -1.17636 (relative 0.02), worst-case bound: -3.13583
- non-inferiority p = 0.9997 (NOT established); power to clear margin if harmless = 92%
- status: HARM  (harm

## 4. INVALID: the experiment never had a chance

Everything above assumes the two arms are comparable. A sample ratio mismatch is direct evidence that they are not: users did not reach the arms in the intended proportion, which points at a bug in assignment, logging, bot filtering or the analysis join. The users who went missing are almost never a random subset, so the comparison is confounded and *no* metric from the experiment can be trusted.

We simulate a broken assignment that sends 49% of users to treatment instead of 50%. The metrics are untouched -- the same true lift is still there -- but the engine refuses to report them, because reporting a metric from an experiment with a known assignment bug invites someone to act on it. The correct response is to fix the pipeline and re-run, not to reason about the numbers.

Note the magnitude: a 1 percentage point deviation looks small, but at 140,458 users it is a chi-square p-value of about 6e-14. That is the point of the check -- at experiment scale, real assignment bugs are unmissable, which is exactly why the gate is set at 0.0005 rather than 0.05 and still essentially never fires on a healthy experiment (validated in `validate_phase3.py`, gate G3d).

In [5]:
df_srm = generate_experiment_data(n_users=2 * n_default, seed=204, treatment_share=0.49)
print(evaluate_experiment(df_srm, default_metric_specs()).render())

# Decision report

## VERDICT: INVALID

- sample ratio mismatch: treatment share 48.9997% vs 50.00% intended (chi-square p = 6.49e-14 < 0.0005)
- the arms are not comparable, so no metric was evaluated; fix assignment or logging and re-run the experiment

alpha = 0.05, guardrail power floor = 80%

### sample ratio check (MISMATCH)
- treatment 68,824 / control 71,634 = 48.9997% vs 50.00% intended
- chi-square = 56.217, p = 6.49e-14



## Takeaways

- The verdicts respond to evidence the way a risk-aware review should: improvement alone is not enough; absence of evidence of harm is not evidence of absence (INVESTIGATE), and established harm vetoes the ship (DO NOT SHIP).
- Guardrails on converter-level mean metrics (like AOV) need far more traffic than the primary conversion test. In practice teams respond by widening the margin, switching to a more sensitive guardrail formulation, or accepting the Type II risk explicitly.
- The error-rate guardrail illustrates why DO-NOT-SHIP triggers on harm BEYOND THE MARGIN, not on any statistically significant harm: at large n, a +0.1pp accepted cost is highly significant but well within tolerance.
- A guardrail's covariate is not a free lunch: CUPED helps only where randomization protects it, which is the full randomized population. Behind a post-treatment filter it is actively harmful, so the engine refuses it structurally rather than by convention.
- SRM is checked first and short-circuits everything. A verdict is only as good as the assumption that the arms are comparable, and that assumption is testable.
- `validate_phase3.py` verifies the machinery: NI test passes ~5% at the margin and ~power at zero harm (G3a), BH controls FDR (G3b), the three scenarios yield their verdicts (G3c), and the SRM gate has the right size and power (G3d).